In [8]:
import pandas as pd
from pathlib import Path
import duckdb

PROJECT = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate")
DB_PATH = PROJECT / "warehouse" / "realestate.duckdb"

path = PROJECT / r"All Data Aggregated in CSV\Delinquint Tax Data HCAD\Hctax_DelNameAdd_March2026.txt"

# --- Fixed-width column specs (0-based, end-exclusive) ---
# Your spec is 1-based inclusive start/end.
# Convert: (start-1, end)
colspecs = [
    (1-1, 13),     # acct
    (15-1, 16),    # class_code
    (18-1, 57),    # name_addr_1
    (59-1, 98),    # name_addr_2
    (100-1, 139),  # name_addr_3
    (141-1, 180),  # name_addr_4
    (182-1, 221),  # name_addr_5
    (223-1, 234),  # mail_zip_12
    (236-1, 275),  # prop_desc_1
    (277-1, 316),  # prop_desc_2
    (318-1, 357),  # prop_desc_3
    (359-1, 398),  # prop_desc_4
    (400-1, 408),  # acreage (9 digits, implied 4 decimals)
    (410-1, 449),  # property_address
    (451-1, 455),  # situs_zip5
    (457-1, 467),  # land_value_11
    (469-1, 479),  # improvement_value_11
    (481-1, 493),  # total_value_13
]

names = [
    "acct",
    "class_code",
    "name_addr_1",
    "name_addr_2",
    "name_addr_3",
    "name_addr_4",
    "name_addr_5",
    "mail_zip_12",
    "prop_desc_1",
    "prop_desc_2",
    "prop_desc_3",
    "prop_desc_4",
    "acreage_raw",
    "property_address",
    "situs_zip5",
    "land_value_raw",
    "improvement_value_raw",
    "total_value_raw",
]

# --- Read fixed width ---
df = pd.read_fwf(
    path,
    colspecs=colspecs,
    names=names,
    dtype="string",
    encoding="latin-1",
)

# --- Basic cleanup ---
for c in names:
    df[c] = df[c].fillna("").str.strip()

# acct must remain 13-digit text (keep leading zeros)
df["acct"] = df["acct"].str.replace(r"\D", "", regex=True).str.zfill(13)

# Zips
df["situs_zip5"] = df["situs_zip5"].str.replace(r"\D", "", regex=True).str[:5]
df["mail_zip5"] = df["mail_zip_12"].str.replace(r"\D", "", regex=True).str[:5]
df["mail_zip4"] = df["mail_zip_12"].str.replace(r"\D", "", regex=True).str[5:9]

# Values: remove non-digits then cast
def to_int(series):
    s = series.str.replace(r"\D", "", regex=True)
    return pd.to_numeric(s, errors="coerce").astype("Int64")

df["land_val"] = to_int(df["land_value_raw"])
df["bld_val"] = to_int(df["improvement_value_raw"])
df["tot_val"] = to_int(df["total_value_raw"])

# Acreage: spec says (5)V9(4) => implied 4 decimals
# Example: 000001234 -> 0.1234 acres (if that’s how it’s encoded)
acre = df["acreage_raw"].str.replace(r"\D", "", regex=True)
df["acreage"] = pd.to_numeric(acre, errors="coerce") / 10_000

print(df.shape)
print(df.head(3))

# --- Save parquet (optional but recommended) ---
out_parquet = PROJECT / "gold" / "tax_delinquency_2026_clean.parquet"
out_parquet.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_parquet, index=False)
print("Saved:", out_parquet)

# --- Load to DuckDB ---
con = duckdb.connect(str(DB_PATH))

con.execute("DROP TABLE IF EXISTS tax_delinquency_2026_clean")
con.execute("""
CREATE TABLE tax_delinquency_2026_clean AS
SELECT * FROM read_parquet(?)
""", [str(out_parquet)])

print(con.execute("SELECT COUNT(*) AS n FROM tax_delinquency_2026_clean").fetchdf())
print(con.execute("DESCRIBE tax_delinquency_2026_clean").fetchdf())
print(con.execute("SELECT acct, class_code, property_address, situs_zip5, tot_val FROM tax_delinquency_2026_clean LIMIT 10").fetchdf())

con.close()

(67307, 24)
            acct class_code                name_addr_1  \
0  0010020000004         F1  MICHAEL RYAN FEAGIN TRUST   
1  0010170000001         F1     120 MILAM HOLDINGS LLC   
2  0010230000010         C2     HARRIS COUNTY ROW DEPT   

                   name_addr_2 name_addr_3 name_addr_4    name_addr_5  \
0              3302 SUFFOLK DR                             HOUSTON TX   
1            7542 DAWN MIST CT                          SUGAR LAND TX   
2  10555 NORTHWEST FWY STE 210                             HOUSTON TX   

  mail_zip_12                 prop_desc_1           prop_desc_2  ...  \
0   770276326  .25 U/D INT IN TR 14 BLK 2                  SSBB  ...   
1   774796324  LT 1 & TRS 2A & 12A BLK 17                  SSBB  ...   
2   770928215    LT 10 BLK 23 (LAND ONLY)  (IMPS*0010230000022)  ...   

  situs_zip5 land_value_raw improvement_value_raw total_value_raw mail_zip5  \
0      77002    00000430203           00000016474   0000000446677     77027   
1      77002   

In [9]:
import duckdb
from pathlib import Path

PROJECT = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate")
DB_PATH = PROJECT / "warehouse" / "realestate.duckdb"

con = duckdb.connect(str(DB_PATH))

print(con.execute("""
SELECT
  COUNT(*) AS delinquent_rows,
  COUNT(DISTINCT d.acct) AS delinquent_accts,
  SUM(CASE WHEN a.acct IS NULL THEN 1 ELSE 0 END) AS missing_in_anchor
FROM tax_delinquency_2026_clean d
LEFT JOIN property_anchor a
  ON d.acct = a.acct
""").fetchdf())

con.close()

   delinquent_rows  delinquent_accts  missing_in_anchor
0            67307             67307            23612.0
